In [ ]:
import numpy as np
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import cmocean.cm as cmo
from matplotlib import rc
from cartopy.crs import Mercator, PlateCarree

from mapgridder import grid_map
from auxdata import get_multibeam_map_W1, get_multibeam_map_W3 
from adcp import distance_to_interface
from depth_pressure_conversions import depth_from_pressure, compute_dynamic_height_profile
from soundspeed import sound_speed_in_DIS_cavity, sound_speed_harmonic_mean
from plot import nice_lonlat_gridlines, scale_bar

rc('font', size=8)
plt.rcParams['font.sans-serif'] = ['Arial'] + plt.rcParams['font.sans-serif'] # Arial as first choice

fig_width = 5.5 
fig_height = 4.65
cbar_aspect = 20

Multibeam

In [ ]:
W1 = get_multibeam_map_W1()
W3 = get_multibeam_map_W3()

Derived map

In [ ]:
adcp_map = pd.read_csv('data/derived/NBP2202_03_ice_draft.csv')

SAR

In [ ]:
sar = xr.load_dataset('data/auxiliary/background/2022-01-21-00_00_2022-01-21-23_59_Sentinel-1_IW_HH_HH_-_decibel_gamma0.nc')

Compare with multibeam

In [ ]:
mb_grid_size = 10 # m
lons = xr.DataArray(adcp_map.longitude.values, dims='z')
lats = xr.DataArray(adcp_map.latitude.values, dims='z')
ground_truth = grid_map(pd.concat([W1,W3], axis=0), mb_grid_size).interp(Lat=lats, Lon=lons)

Compare with using beam average

In [ ]:
def ice_draft_from_beam_average(echo_intensity_file):
    ds = xr.open_dataset(echo_intensity_file)
    ds['distance_to_interface'] = distance_to_interface(ds.mean(dim='beam'), 
                                                        interpolate=True, 
                                                        range = 'range',
                                                        rmin=200, 
                                                        rmax=1150,  
                                                        ampmin=100,
                                                       )  
    ds['depth'] = depth_from_pressure(ds['pressure'], ds['latitude'], compute_dynamic_height_profile())
    c_profile = sound_speed_in_DIS_cavity(make_plot=False)
    c_correction_factor = np.zeros(ds.sizes['time'])
    for t in range(ds.sizes['time']):
        d = ds.depth.isel(time=t)
        c_av = sound_speed_harmonic_mean(d, d-ds.distance_to_interface.isel(time=t), c_profile)              
        c_correction_factor[t] = c_av/ds.sound_speed.isel(time=t)
    ds['correction_factor'] = (('time'), c_correction_factor)
    
    ds['ice_draft'] = ds.depth - ds.distance_to_interface*ds.correction_factor    

    return pd.DataFrame({'longitude' : ds.longitude.values, 
                         'latitude' : ds.latitude.values, 
                         'ice_draft'   : ds.ice_draft.values})

adcp_map_av = ice_draft_from_beam_average('data/derived/NBP2202_03_cleaned.nc')
lons_av = xr.DataArray(adcp_map_av.longitude.values, dims='z')
lats_av = xr.DataArray(adcp_map_av.latitude.values, dims='z')
ground_truth_av = grid_map(pd.concat([W1,W3], axis=0), mb_grid_size).interp(Lat=lats_av, Lon=lons_av)

### Make figure

In [ ]:
lon_min = -113.38
lon_max = -113.07
lat_max = -74.17
lat_min = -74.243

vmin=0
vmax=200
cmap=cmo.haline
step=1

proj = Mercator(central_longitude=-112.75,
                min_latitude = -75,
                max_latitude = -73,
                latitude_true_scale = -74.2)

In [ ]:
fig = plt.figure(figsize=(fig_width, fig_height*1.6))
gs = GridSpec(4, 2, figure=fig, height_ratios = (1,1,1,0.05), wspace=0.03, hspace=0.03)

grid_lats = [-74.24, -74.22, -74.2, -74.18]
grid_lons = [-113.35, -113.25, -113.15]

lolasize = 6

step = 1
plot_sar = True

# ADCP map
ax = fig.add_subplot(gs[0,0], projection = proj)
if plot_sar:
    ax.pcolormesh(sar.lon, sar.lat, sar.Band1, cmap=cmo.gray, transform = PlateCarree(), zorder=-11)
im = ax.scatter(adcp_map.longitude[::step], adcp_map.latitude[::step], c=adcp_map.ice_draft[::step], 
                transform=PlateCarree(), 
                s=1, marker='.', vmin=vmin, vmax=vmax, cmap=cmap)
ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=PlateCarree())
gl = nice_lonlat_gridlines(ax, size=lolasize, zorder=-10, alpha=0.2, labels = ['left', 'top'], longitudes = grid_lons, latitudes = grid_lats)
ax.annotate('(a)', xy=(0.01, 0.93), xycoords='axes fraction', color = 'w', weight='bold')
# Mark foot print size
ax.scatter(lon_min + 0.01, lat_min + 0.003, s=3, c='k', transform=PlateCarree())
ax.text(lon_min + 0.015, lat_min + 0.003, 'footprint', ha='left', va='center', c='k', size=8, transform=PlateCarree())

# Multibeam map
ax = fig.add_subplot(gs[0,1], projection = proj)
if plot_sar:
    ax.pcolormesh(sar.lon, sar.lat, sar.Band1, cmap=cmo.gray, transform = PlateCarree(), zorder=-11)
for map in [W1, W3]:
    im = ax.scatter(map.Lon[::step], map.Lat[::step], c=map.D[::step], transform=PlateCarree(), 
                    s=0.1, marker='.', vmin=vmin, vmax=vmax, cmap=cmap)
ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=PlateCarree())
gl = nice_lonlat_gridlines(ax, size=lolasize, zorder=-10, alpha=0.2, labels = ['top'], longitudes = grid_lons, latitudes = grid_lats)
ax.annotate('(b)', xy=(0.01, 0.93), xycoords='axes fraction', color = 'w', weight='bold')
scale_bar(ax, length=1, location = (0.03,0.03), textoffset=80, fontsize=6, linewidth=2)

# Both
ax = fig.add_subplot(gs[1,0], projection = proj)
if plot_sar:
    ax.pcolormesh(sar.lon, sar.lat, sar.Band1, cmap=cmo.gray, transform = PlateCarree(), zorder=-11)
for map in [W1, W3]:
    im = ax.scatter(map.Lon[::step], map.Lat[::step], c=map.D[::step], transform=PlateCarree(), 
                    s=0.1, marker='.', vmin=vmin, vmax=vmax, cmap=cmap)
im = ax.scatter(adcp_map.longitude[::step], adcp_map.latitude[::step], c=adcp_map.ice_draft[::step], transform=PlateCarree(), 
                s=1, marker='.', vmin=vmin, vmax=vmax, cmap=cmap)
ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=PlateCarree())
gl = nice_lonlat_gridlines(ax, size=lolasize, zorder=-10, alpha=0.2, labels = ['left'], longitudes = grid_lons, latitudes = grid_lats)
ax.annotate('(c)', xy=(0.01, 0.93), xycoords='axes fraction', color = 'w', weight='bold')

# Difference
ax = fig.add_subplot(gs[1,1], projection = proj)
if plot_sar:
    ax.pcolormesh(sar.lon, sar.lat, sar.Band1, cmap=cmo.gray, transform = PlateCarree(), zorder=-11)
diff = ax.scatter(adcp_map.longitude[::step], adcp_map.latitude[::step], c=adcp_map.ice_draft[::step] - ground_truth[::step], 
                  transform=PlateCarree(), 
                  s=1, marker='.', vmin=-30, vmax=30, cmap=cmo.balance)
ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=PlateCarree())
gl = nice_lonlat_gridlines(ax, size=lolasize, zorder=-10, alpha=0.2, labels = [], longitudes = grid_lons, latitudes = grid_lats)
ax.annotate('(d)', xy=(0.01, 0.93), xycoords='axes fraction', color = 'w', weight='bold')

# Beam average
ax = fig.add_subplot(gs[2,0], projection = proj)
if plot_sar:
    ax.pcolormesh(sar.lon, sar.lat, sar.Band1, cmap=cmo.gray, transform = PlateCarree(), zorder=-11)
im = ax.scatter(adcp_map_av.longitude[::step], adcp_map_av.latitude[::step], c=adcp_map_av.ice_draft[::step], transform=PlateCarree(), 
                s=20, marker='.', vmin=vmin, vmax=vmax, cmap=cmap)
ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=PlateCarree())
gl = nice_lonlat_gridlines(ax, size=lolasize, zorder=-10, alpha=0.2, labels = ['left'], longitudes = grid_lons, latitudes = grid_lats)
ax.annotate('(e)', xy=(0.01, 0.93), xycoords='axes fraction', color = 'w', weight='bold')
# Mark foot print size
ax.scatter(lon_min + 0.02 , lat_min + 0.005, s=60, c='k', transform=PlateCarree())
ax.text(lon_min + 0.035, lat_min + 0.005, 'footprint', ha='left', va='center_baseline', c='k', size=8, transform=PlateCarree())

# Beam average difference
ax = fig.add_subplot(gs[2,1], projection = proj)
if plot_sar:
    ax.pcolormesh(sar.lon, sar.lat, sar.Band1, cmap=cmo.gray, transform = PlateCarree(), zorder=-11)
diff = ax.scatter(adcp_map_av.longitude[::step], adcp_map_av.latitude[::step], c=adcp_map_av.ice_draft[::step] - ground_truth_av[::step], 
                  transform=PlateCarree(), 
                  s=10, marker='.', vmin=-30, vmax=30, cmap=cmo.balance)
ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=PlateCarree())
gl = nice_lonlat_gridlines(ax, size=lolasize, zorder=-10, alpha=0.2, labels = [], longitudes = grid_lons, latitudes = grid_lats)
ax.annotate('(f)', xy=(0.01, 0.93), xycoords='axes fraction', color = 'w', weight='bold')

## colorbars
# ice draft
cax = fig.add_subplot(gs[3,0])
cbar = fig.colorbar(im, cax=cax, extend='both', aspect = cbar_aspect, orientation='horizontal', shrink=0.9)
cbar.ax.tick_params(labelsize=6, length=1.5)
cbar.set_label( label= 'Ice draft (m)', size=7)

# difference
cax = fig.add_subplot(gs[3,1])
cbar = fig.colorbar(diff, cax=cax, extend='both', aspect = cbar_aspect, orientation='horizontal', shrink=0.9)
cbar.ax.tick_params(labelsize=6, length=1.5)
cbar.set_label( label= 'Difference (m)', size=7)

# Adjust distance between plots
fig.subplots_adjust(wspace=0, hspace=0)

plt.savefig('figures/fig6.png', bbox_inches = 'tight', dpi=400)

### Version for presentations

In [ ]:
rc('font', size=9)


fig = plt.figure(figsize=(fig_height*2.1, fig_width))
gs = GridSpec(2,4, figure=fig, width_ratios = (1,1,1,0.05), hspace=0.03, wspace=0.03)

grid_lats = [-74.24, -74.22, -74.2, -74.18]
grid_lons = [-113.35, -113.25, -113.15]

lolasize = 6

step = 1
plot_sar = True

# Multibeam
ax = fig.add_subplot(gs[0,0], projection = proj)
if plot_sar:
    ax.pcolormesh(sar.lon, sar.lat, sar.Band1, cmap=cmo.gray, transform = PlateCarree(), zorder=-11)
for map in [W1, W3]:
    im = ax.scatter(map.Lon[::step], map.Lat[::step], c=map.D[::step], transform=PlateCarree(), 
                    s=0.1, marker='.', vmin=vmin, vmax=vmax, cmap=cmap)
ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=PlateCarree())
gl = nice_lonlat_gridlines(ax, zorder=-10, alpha=0.2, labels = ['top', 'left'], longitudes = grid_lons, latitudes = grid_lats)
scale_bar(ax, length=1, location = (0.03,0.03), textoffset=80, linewidth=2)

# Overlap
ax = fig.add_subplot(gs[1,0], projection = proj)
if plot_sar:
    ax.pcolormesh(sar.lon, sar.lat, sar.Band1, cmap=cmo.gray, transform = PlateCarree(), zorder=-11)
for map in [W1, W3]:
    im = ax.scatter(map.Lon[::step], map.Lat[::step], c=map.D[::step], transform=PlateCarree(), 
                    s=0.1, marker='.', vmin=vmin, vmax=vmax, cmap=cmap)
im = ax.scatter(adcp_map.longitude[::step], adcp_map.latitude[::step], c=adcp_map.ice_draft[::step], transform=PlateCarree(), 
                s=1, marker='.', vmin=vmin, vmax=vmax, cmap=cmap)
ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=PlateCarree())
gl = nice_lonlat_gridlines(ax, zorder=-10, alpha=0.2, labels = ['left', 'bottom'], longitudes = grid_lons, latitudes = grid_lats)

# ADCP
ax = fig.add_subplot(gs[0,1], projection = proj)
if plot_sar:
    ax.pcolormesh(sar.lon, sar.lat, sar.Band1, cmap=cmo.gray, transform = PlateCarree(), zorder=-11)
im = ax.scatter(adcp_map.longitude[::step], adcp_map.latitude[::step], c=adcp_map.ice_draft[::step], transform=PlateCarree(), 
                s=1, marker='.', vmin=vmin, vmax=vmax, cmap=cmap)
ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=PlateCarree())
gl = nice_lonlat_gridlines(ax, zorder=-10, alpha=0.2, labels = ['top'], longitudes = grid_lons, latitudes = grid_lats)
# Mark foot print size
ax.scatter(lon_min + 0.01, lat_min + 0.003, s=3, c='k', transform=PlateCarree())
ax.text(lon_min + 0.015, lat_min + 0.003, 'footprint', ha='left', va='center', c='k', transform=PlateCarree())

# Difference
ax = fig.add_subplot(gs[1,1], projection = proj)
if plot_sar:
    ax.pcolormesh(sar.lon, sar.lat, sar.Band1, cmap=cmo.gray, transform = PlateCarree(), zorder=-11)
diff = ax.scatter(adcp_map.longitude[::step], adcp_map.latitude[::step], c=adcp_map.ice_draft[::step] - ground_truth[::step], 
                  transform=PlateCarree(), 
                  s=1, marker='.', vmin=-30, vmax=30, cmap=cmo.balance)
ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=PlateCarree())
gl = nice_lonlat_gridlines(ax, zorder=-10, alpha=0.2, labels = ['bottom'], longitudes = grid_lons, latitudes = grid_lats)

# ADCP average
ax = fig.add_subplot(gs[0,2], projection = proj)
if plot_sar:
    ax.pcolormesh(sar.lon, sar.lat, sar.Band1, cmap=cmo.gray, transform = PlateCarree(), zorder=-11)
im = ax.scatter(adcp_map_av.longitude[::step], adcp_map_av.latitude[::step], c=adcp_map_av.ice_draft[::step], transform=PlateCarree(), 
                s=20, marker='.', vmin=vmin, vmax=vmax, cmap=cmap)
ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=PlateCarree())
gl = nice_lonlat_gridlines(ax, zorder=-10, alpha=0.2, labels = ['top'], longitudes = grid_lons, latitudes = grid_lats)
# Mark foot print size
ax.scatter(lon_min + 0.02 , lat_min + 0.005, s=60, c='k', transform=PlateCarree())
ax.text(lon_min + 0.035, lat_min + 0.005, 'footprint', ha='left', va='center_baseline', c='k', transform=PlateCarree())


# Difference
ax = fig.add_subplot(gs[1,2], projection = proj)
if plot_sar:
    ax.pcolormesh(sar.lon, sar.lat, sar.Band1, cmap=cmo.gray, transform = PlateCarree(), zorder=-11)
diff = ax.scatter(adcp_map_av.longitude[::step], adcp_map_av.latitude[::step], c=adcp_map_av.ice_draft[::step] - ground_truth_av[::step], 
                  transform=PlateCarree(), 
                  s=10, marker='.', vmin=-30, vmax=30, cmap=cmo.balance)
ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=PlateCarree())
gl = nice_lonlat_gridlines(ax,  zorder=-10, alpha=0.2, labels = ['bottom'], longitudes = grid_lons, latitudes = grid_lats)

## colorbars
# ice draft
cax = fig.add_subplot(gs[0,3])
cbar = fig.colorbar(im, cax=cax, extend='both', aspect = cbar_aspect, orientation='vertical', shrink=0.9)
cbar.ax.tick_params(length=1.5)
cbar.set_label( label= 'Ice draft (m)')

# difference
cax = fig.add_subplot(gs[1,3])
cbar = fig.colorbar(diff, cax=cax, extend='both', aspect = cbar_aspect, orientation='vertical', shrink=0.9)
cbar.ax.tick_params(length=1.5)
cbar.set_label( label= 'Difference (m)')

plt.savefig('figures/fig6_landscape.png', bbox_inches = 'tight', dpi=400)